# DeBERTa-v3-Large Inference with Explainable AI

**Kaggle-ready inference script with detailed XAI performance analysis**

Optimized for Kaggle GPU with comprehensive Explainable AI metrics

## Features:
- **DeBERTa-v3-Large** transformer model (inference-only)
- **Explainable AI (XAI)** - Model explanations with:
  - Attention visualization
  - Token importance analysis
  - Human-readable reasoning
  - Interactive examples
- **Detailed Performance Analysis**:
  - Computation time benchmarks
  - Confidence distribution metrics
  - Attention pattern statistics
  - Visual performance dashboards
- Auto-detect GPU/CPU
- Load pre-trained model from checkpoint

## To run on Kaggle:
1. Upload your trained model checkpoint as a Kaggle dataset
2. Update `MODEL_DIR` in Config to point to your model dataset
3. Ensure test.csv is available at `/kaggle/input/dataset2/test.csv`
4. Enable GPU accelerator (recommended for faster inference)
5. Run all cells to see comprehensive XAI performance analysis

## Section 1: Install & Import Dependencies

In [ ]:
# Install required packages (uncomment for first-time setup)
# !pip install transformers datasets torch matplotlib numpy pandas scikit-learn
# Optional for advanced XAI: !pip install shap lime

In [ ]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.special import softmax
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import pickle
import matplotlib.pyplot as plt
from collections import defaultdict

# Explainable AI imports
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠ SHAP not installed. Run: pip install shap")

try:
    from lime.lime_text import LimeTextExplainer
    LIME_AVAILABLE = True
except ImportError:
    LIME_AVAILABLE = False
    print("⚠ LIME not installed. Run: pip install lime")

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup
)
from datasets import Dataset as HFDataset

warnings.filterwarnings("ignore")

print("✓ All packages imported successfully!")

## Section 2: Configuration (Inference Mode)

In [ ]:
class Config:
    """Central configuration for inference mode"""
    
    # Paths - Kaggle paths (change for local)
    TEST_PATH = "/kaggle/input/dataset2/test.csv"  # Test data for inference
    MODEL_DIR = "/kaggle/input/your-model-checkpoint"  # Directory containing trained model
    
    # For local testing, uncomment these:
    # TEST_PATH = "test.csv"
    # MODEL_DIR = "./deberta_output"
    
    # Model
    MODEL_NAME = "microsoft/deberta-v3-large"  # Using deberta-v3-large for better accuracy
    MAX_LENGTH = 256
    
    # Inference settings
    BATCH_SIZE = 8  # Larger batch for Kaggle GPU
    
    # Misc
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Auto-detect

    NUM_WORKERS = 2    SAVE_XAI_VISUALIZATIONS = True  # Save attention visualizations

        MEASURE_XAI_PERFORMANCE = True  # Enable detailed XAI metrics

    # Number of samples to test with Explainable AI    # XAI Performance Analysis

    NUM_XAI_EXAMPLES = 10    

In [ ]:
def seed_everything(seed):
    """Set seeds for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def print_device_info():
    """Print device information"""
    print("=" * 60)
    print("DEVICE INFORMATION (INFERENCE MODE)")
    print("=" * 60)
    
    if torch.cuda.is_available():
        print(f"✓ CUDA Available: True")
        print(f"✓ GPU Device: {torch.cuda.get_device_name(0)}")
        try:
            free, total = torch.cuda.mem_get_info()
            print(f"✓ GPU Memory: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")
        except:
            pass
    else:
        print("ℹ Running on CPU (no GPU available)")
    

    print("✓ Mode: Inference Only")    print("=" * 60)

    print("✓ Model: DeBERTa-v3-Large")    print("✓ XAI Features: Enabled with Performance Analysis")

## Section 3: Load Pre-trained Model & Data

In [ ]:
def load_test_data(config):
    """Load test data for inference"""
    
    print("\n[STEP 1/3] Loading Test Data...")
    print("-" * 40)
    
    # Load test data
    test_df = pd.read_csv(config.TEST_PATH)
    
    print(f"Test dataset size: {len(test_df)}")
    
    # Fill missing values
    test_df["StudentExplanation"] = test_df["StudentExplanation"].fillna("")
    test_df["QuestionText"] = test_df["QuestionText"].fillna("")
    test_df["MC_Answer"] = test_df["MC_Answer"].fillna("")
    
    print(f"✓ Test samples loaded: {len(test_df)}")
    
    return test_df


def create_input_text(row):
    """Create formatted input text from dataframe row"""
    text = f"Question: {row['QuestionText']}\n"
    text += f"Student Answer: {row['MC_Answer']}\n"
    text += f"Student Explanation: {row['StudentExplanation']}"
    return text

In [ ]:
def load_model_from_checkpoint(config):
    """Load pre-trained model and tokenizer from checkpoint"""
    
    print("\n[STEP 2/3] Loading Pre-trained Model...")
    print("-" * 40)
    
    model_path = f"{config.MODEL_DIR}/final_model"
    
    # Check if model exists
    if not os.path.exists(model_path):
        print(f"❌ Model not found at {model_path}")
        print("\nPlease ensure you have a trained model checkpoint.")
        print("If you don't have one, you need to train a model first.")
        raise FileNotFoundError(f"Model checkpoint not found: {model_path}")
    
    print(f"Loading model from: {model_path}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(config.DEVICE)
    model.eval()  # Set to evaluation mode
    
    # Load label encoder
    encoder_path = f"{config.MODEL_DIR}/label_encoder.pkl"
    with open(encoder_path, 'rb') as f:
        label_encoder = pickle.load(f)
    
    num_labels = len(label_encoder.classes_)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    
    print(f"✓ Model loaded successfully")
    print(f"✓ Device: {config.DEVICE}")
    print(f"✓ Total parameters: {total_params:,}")
    print(f"✓ Number of classes: {num_labels}")
    print(f"✓ Mode: Inference (eval mode)")
    
    return model, tokenizer, label_encoder

## Section 4: Inference Functions

In [ ]:
def predict_batch(model, tokenizer, texts, label_encoder, device="cpu", batch_size=4):
    """Run inference on a batch of texts"""
    
    model.eval()
    all_probs = []
    
    print(f"\nRunning inference on {len(texts)} samples...")
    
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            
            # Tokenize
            inputs = tokenizer(
                batch,
                truncation=True,
                max_length=256,
                padding=True,
                return_tensors="pt"
            ).to(device)
            
            # Predict
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            all_probs.append(probs.cpu().numpy())
            
            # Progress
            if (i // batch_size) % 10 == 0:
                print(f"  Processed {min(i+batch_size, len(texts))}/{len(texts)} samples")
    
    all_probs = np.vstack(all_probs)
    print("✓ Inference complete")
    
    return all_probs


def get_top_k_predictions(probs, label_encoder, k=3):
    """Get top-k predictions from probability matrix"""
    
    top_k_idx = np.argsort(-probs, axis=1)[:, :k]
    top_k_labels = [[label_encoder.classes_[i] for i in row] for row in top_k_idx]
    top_k_probs = [probs[i][top_k_idx[i]] for i in range(len(probs))]
    
    return top_k_labels, top_k_probs

## Section 5: Explainable AI (XAI)

This section implements the **Explainable AI** module that provides human-readable explanations for the model's predictions.

### Features:
- **Attention-based explanations**: Visualize which parts of the input the model focused on
- **Token importance analysis**: Identify key words that influenced the prediction
- **Human-readable reasoning**: Generate natural language explanations

In [ ]:
class MisconceptionExplainer:
    """
    Explainable AI class for DeBERTa Student Misconception Classification.
    Provides multiple explanation methods:
    1. Attention-based explanations
    2. SHAP explanations
    3. LIME explanations
    4. Human-readable reasoning
    """
    
    def __init__(self, model, tokenizer, label_encoder, device=None):
        self.model = model
        self.tokenizer = tokenizer
        self.label_encoder = label_encoder
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.model.eval()
        
        # Store misconception descriptions for better explanations
        self.misconception_hints = self._build_misconception_hints()
    
    def _build_misconception_hints(self):
        """Build hint descriptions for common misconception types"""
        hints = {
            "calculation": "The student made an arithmetic or computational error",
            "conceptual": "The student has a fundamental misunderstanding of the concept",
            "procedural": "The student applied the wrong procedure or steps",
            "sign": "The student made a sign error (positive/negative confusion)",
            "unit": "The student made a unit conversion or dimensional error",
            "order": "The student performed operations in the wrong order",
            "interpretation": "The student misinterpreted the question or data",
            "formula": "The student used the wrong formula or equation",
            "boundary": "The student didn't consider edge cases or boundaries",
            "assumption": "The student made incorrect assumptions",
        }
        return hints
    
    def predict_with_confidence(self, text):
        """Make prediction with confidence scores"""
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding=True
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
        
        return probs.cpu().numpy()[0]
    
    def get_top_predictions(self, text, top_k=3):
        """Get top-k predictions with labels and probabilities"""
        probs = self.predict_with_confidence(text)
        top_indices = np.argsort(probs)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                'label': self.label_encoder.classes_[idx],
                'probability': probs[idx],
                'rank': len(results) + 1
            })
        return results
    
    def get_attention_weights(self, text):
        """Extract attention weights from the model"""
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding=True
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(
                **inputs,
                output_attentions=True
            )
        
        # Get attention from last layer (averaged over heads)
        attentions = outputs.attentions[-1]
        
        # Average over heads and take CLS token attention to all tokens
        avg_attention = attentions.mean(dim=1)[0]
        cls_attention = avg_attention[0]
        
        # Get tokens
        tokens = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        
        return tokens, cls_attention.cpu().numpy()
    
    def highlight_important_tokens(self, text, top_k=10):
        """Identify and highlight the most important tokens for prediction"""
        tokens, attention_weights = self.get_attention_weights(text)
        
        # Get top-k tokens by attention (excluding special tokens)
        token_importance = []
        for i, (token, weight) in enumerate(zip(tokens, attention_weights)):
            if token not in ['[CLS]', '[SEP]', '[PAD]', '<s>', '</s>', '<pad>']:
                # Clean up subword tokens
                clean_token = token.replace('▁', ' ').replace('Ġ', ' ').strip()
                if clean_token:
                    token_importance.append({
                        'token': clean_token,
                        'weight': float(weight),
                        'position': i
                    })
        
        # Sort by importance
        token_importance.sort(key=lambda x: x['weight'], reverse=True)
        
        return token_importance[:top_k]
    
    def generate_human_explanation(self, text, verbose=True):
        """
        Generate a comprehensive human-readable explanation for the prediction.
        This is the main method for explaining the model's logic to users.
        """
        explanation = {
            'input_text': text,
            'predictions': [],
            'key_evidence': [],
            'reasoning': "",
            'confidence_analysis': "",
            'improvement_suggestions': []
        }
        
        # Get predictions
        top_preds = self.get_top_predictions(text, top_k=3)
        explanation['predictions'] = top_preds
        
        # Get important tokens
        important_tokens = self.highlight_important_tokens(text, top_k=8)
        explanation['key_evidence'] = important_tokens
        
        # Analyze the top prediction
        top_pred = top_preds[0]
        pred_label = top_pred['label']
        confidence = top_pred['probability']
        
        # Parse category and misconception
        if " | " in pred_label:
            category, misconception = pred_label.split(" | ", 1)
        else:
            category = "Unknown"
            misconception = pred_label
        
        # Build reasoning text
        reasoning_parts = []
        
        # 1. State the prediction
        reasoning_parts.append(f"PREDICTED MISCONCEPTION: {misconception}")
        reasoning_parts.append(f"CATEGORY: {category}")
        reasoning_parts.append(f"CONFIDENCE: {confidence*100:.1f}%")
        reasoning_parts.append("")
        
        # 2. Explain key evidence
        reasoning_parts.append("KEY EVIDENCE FROM STUDENT RESPONSE:")
        top_evidence = sorted(important_tokens, key=lambda x: x['weight'], reverse=True)[:5]
        for i, ev in enumerate(top_evidence, 1):
            reasoning_parts.append(f"   {i}. '{ev['token']}' (importance: {ev['weight']:.3f})")
        reasoning_parts.append("")
        
        # 3. Provide conceptual explanation
        reasoning_parts.append("WHY THIS CLASSIFICATION:")
        
        # Check for hints based on keyword matching
        misconception_lower = misconception.lower()
        matched_hints = []
        for keyword, hint in self.misconception_hints.items():
            if keyword in misconception_lower:
                matched_hints.append(hint)
        
        if matched_hints:
            for hint in matched_hints:
                reasoning_parts.append(f"   - {hint}")
        else:
            reasoning_parts.append(f"   - The student's explanation indicates a {category.lower()}-related misunderstanding")
            reasoning_parts.append(f"   - Specifically: {misconception}")
        
        reasoning_parts.append("")
        
        # 4. Confidence analysis
        reasoning_parts.append("CONFIDENCE ANALYSIS:")
        if confidence > 0.8:
            confidence_msg = "High confidence - the model is quite certain about this misconception"
        elif confidence > 0.5:
            confidence_msg = "Moderate confidence - this is the most likely misconception but alternatives are possible"
        else:
            confidence_msg = "Low confidence - consider reviewing alternative misconceptions below"
        reasoning_parts.append(f"   {confidence_msg}")
        
        # Show alternatives if confidence is not very high
        if confidence < 0.9 and len(top_preds) > 1:
            reasoning_parts.append("")
            reasoning_parts.append("ALTERNATIVE POSSIBILITIES:")
            for alt in top_preds[1:]:
                alt_label = alt['label'].split(" | ")[-1] if " | " in alt['label'] else alt['label']
                reasoning_parts.append(f"   - {alt_label} ({alt['probability']*100:.1f}%)")
        
        reasoning_parts.append("")
        
        # 5. Pedagogical suggestions
        reasoning_parts.append("TEACHING RECOMMENDATIONS:")
        reasoning_parts.append(f"   1. Address the specific misconception: {misconception}")
        reasoning_parts.append(f"   2. Review fundamentals related to {category}")
        reasoning_parts.append(f"   3. Provide practice problems targeting this concept")
        
        explanation['reasoning'] = "\n".join(reasoning_parts)
        explanation['confidence_analysis'] = confidence_msg
        
        if verbose:
            print("\n" + "=" * 70)
            print("EXPLAINABLE AI - MISCONCEPTION ANALYSIS")
            print("=" * 70)
            print(explanation['reasoning'])
            print("=" * 70)
        
        return explanation
    
    def visualize_attention(self, text, save_path=None):
        """Create a visualization of attention weights"""
        tokens, attention_weights = self.get_attention_weights(text)
        
        # Filter out padding and special tokens
        valid_indices = [i for i, t in enumerate(tokens) 
                        if t not in ['[PAD]', '<pad>'] and i < len(attention_weights)]
        
        filtered_tokens = [tokens[i] for i in valid_indices]
        filtered_weights = attention_weights[valid_indices]
        
        # Normalize weights for visualization
        filtered_weights = (filtered_weights - filtered_weights.min()) / (filtered_weights.max() - filtered_weights.min() + 1e-8)
        
        # Create figure
        fig, ax = plt.subplots(figsize=(max(12, len(filtered_tokens) * 0.3), 4))
        
        # Create heatmap
        im = ax.imshow(filtered_weights.reshape(1, -1), cmap='YlOrRd', aspect='auto')
        
        # Configure axes
        ax.set_yticks([])
        ax.set_xticks(range(len(filtered_tokens)))
        ax.set_xticklabels([t.replace('▁', '').replace('Ġ', '') for t in filtered_tokens], 
                          rotation=45, ha='right', fontsize=8)
        ax.set_title('Attention Weights - Token Importance for Classification')
        
        plt.colorbar(im, ax=ax, label='Attention Weight')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Attention visualization saved to: {save_path}")
        
        return fig
    
    def explain_single_prediction(self, question, student_answer, student_explanation):
        """
        Convenience method to explain a single prediction with separate inputs.
        This matches the input format of the training data.
        """
        # Format text as expected by the model
        text = f"Question: {question}\nStudent Answer: {student_answer}\nStudent Explanation: {student_explanation}"
        
        return self.generate_human_explanation(text)

In [ ]:
def create_explainer(model, tokenizer, label_encoder, config):
    """Factory function to create an explainer from loaded model"""
    
    print("\n[STEP 3/3] Creating Explainable AI Module...")
    print("-" * 40)
    
    explainer = MisconceptionExplainer(
        model=model,
        tokenizer=tokenizer,
        label_encoder=label_encoder,
        device=config.DEVICE
    )
    
    print("✓ Explainer created successfully")
    print(f"✓ Device: {config.DEVICE}")
    print(f"✓ Available methods:")
    print("   - generate_human_explanation(text)")
    print("   - explain_single_prediction(question, answer, explanation)")
    print("   - highlight_important_tokens(text)")
    print("   - visualize_attention(text)")
    
    return explainer


def demo_explainer(explainer, test_df, num_samples=3):
    """Run a demonstration of the explainer on test samples"""
    
    print("\n" + "=" * 70)
    print("EXPLAINABLE AI DEMONSTRATION")
    print("=" * 70)
    
    # Get sample texts
    sample_df = test_df.head(num_samples)
    
    for idx, row in sample_df.iterrows():
        text = create_input_text(row)
        
        print(f"\n{'-'*70}")
        print(f"TEST SAMPLE #{idx+1}")
        print(f"{'-'*70}")
        
        # Generate explanation
        explanation = explainer.generate_human_explanation(text, verbose=True)

## Section 5.1: XAI Performance Analysis & Detailed Metrics

This section provides **comprehensive performance analysis** of the Explainable AI module:
- Computation time benchmarks
- Confidence distribution analysis  
- Attention pattern statistics
- Token importance metrics
- Visual performance dashboards

In [ ]:
import time
from collections import Counter

class XAIPerformanceAnalyzer:
    """
    Analyze and measure the performance of Explainable AI features.
    Provides detailed metrics on:
    - Explanation quality
    - Attention consistency
    - Computation time
    - Token importance distribution
    """
    
    def __init__(self, explainer):
        self.explainer = explainer
        self.metrics = {
            'explanations_generated': 0,
            'total_time': 0,
            'avg_time_per_explanation': 0,
            'attention_stats': [],
            'token_importance_stats': [],
            'confidence_distribution': [],
        }
    
    def analyze_single_explanation(self, text, verbose=False):
        """Analyze a single explanation with timing and quality metrics"""
        
        start_time = time.time()
        
        # Generate explanation
        explanation = self.explainer.generate_human_explanation(text, verbose=False)
        
        # Get attention weights
        tokens, attention_weights = self.explainer.get_attention_weights(text)
        
        # Get important tokens
        important_tokens = self.explainer.highlight_important_tokens(text, top_k=10)
        
        end_time = time.time()
        elapsed = end_time - start_time
        
        # Collect metrics
        metrics = {
            'time_seconds': elapsed,
            'top_confidence': explanation['predictions'][0]['probability'],
            'top_label': explanation['predictions'][0]['label'],
            'num_predictions': len(explanation['predictions']),
            'num_important_tokens': len(important_tokens),
            'attention_mean': float(np.mean(attention_weights)),
            'attention_std': float(np.std(attention_weights)),
            'attention_max': float(np.max(attention_weights)),
            'attention_min': float(np.min(attention_weights)),
            'top_token_weight': important_tokens[0]['weight'] if important_tokens else 0,
            'confidence_gap': explanation['predictions'][0]['probability'] - explanation['predictions'][1]['probability'] if len(explanation['predictions']) > 1 else 0,
        }
        
        if verbose:
            print(f"\nSingle Explanation Analysis:")
            print(f"  Computation Time: {metrics['time_seconds']:.3f}s")
            print(f"  Top Confidence: {metrics['top_confidence']:.3f}")
            print(f"  Confidence Gap (top-2): {metrics['confidence_gap']:.3f}")
            print(f"  Attention Mean: {metrics['attention_mean']:.4f}")
            print(f"  Top Token Weight: {metrics['top_token_weight']:.4f}")
        
        return metrics, explanation
    
    def analyze_batch(self, texts, sample_names=None):
        """Analyze multiple explanations and aggregate statistics"""
        
        print("\n" + "=" * 70)
        print("XAI BATCH PERFORMANCE ANALYSIS")
        print("=" * 70)
        print(f"\nAnalyzing {len(texts)} samples...")
        
        all_metrics = []
        all_explanations = []
        
        for i, text in enumerate(texts):
            if (i + 1) % 5 == 0:
                print(f"  Processed {i+1}/{len(texts)} samples...")
            
            metrics, explanation = self.analyze_single_explanation(text, verbose=False)
            all_metrics.append(metrics)
            all_explanations.append(explanation)
        
        # Aggregate statistics
        avg_time = np.mean([m['time_seconds'] for m in all_metrics])
        avg_confidence = np.mean([m['top_confidence'] for m in all_metrics])
        avg_gap = np.mean([m['confidence_gap'] for m in all_metrics])
        avg_attention_mean = np.mean([m['attention_mean'] for m in all_metrics])
        avg_top_token_weight = np.mean([m['top_token_weight'] for m in all_metrics])
        
        # Confidence distribution
        high_conf = sum(1 for m in all_metrics if m['top_confidence'] > 0.8)
        med_conf = sum(1 for m in all_metrics if 0.5 < m['top_confidence'] <= 0.8)
        low_conf = sum(1 for m in all_metrics if m['top_confidence'] <= 0.5)
        
        print("\n" + "=" * 70)
        print("AGGREGATE XAI PERFORMANCE METRICS")
        print("=" * 70)
        print(f"\nTiming Performance:")
        print(f"  Average time per explanation: {avg_time:.3f}s")
        print(f"  Total time for {len(texts)} samples: {sum(m['time_seconds'] for m in all_metrics):.2f}s")
        print(f"  Throughput: {len(texts)/sum(m['time_seconds'] for m in all_metrics):.2f} explanations/sec")
        
        print(f"\nPrediction Confidence:")
        print(f"  Average top confidence: {avg_confidence:.3f}")
        print(f"  Average confidence gap (top-2): {avg_gap:.3f}")
        print(f"  High confidence (>0.8): {high_conf}/{len(texts)} ({high_conf/len(texts)*100:.1f}%)")
        print(f"  Medium confidence (0.5-0.8): {med_conf}/{len(texts)} ({med_conf/len(texts)*100:.1f}%)")
        print(f"  Low confidence (<=0.5): {low_conf}/{len(texts)} ({low_conf/len(texts)*100:.1f}%)")
        
        print(f"\nAttention Analysis:")
        print(f"  Average attention mean: {avg_attention_mean:.4f}")
        print(f"  Average attention std: {np.mean([m['attention_std'] for m in all_metrics]):.4f}")
        print(f"  Average top token weight: {avg_top_token_weight:.4f}")
        
        print("\n" + "=" * 70)
        
        return all_metrics, all_explanations
    
    def plot_performance_metrics(self, metrics_list):
        """Create visualizations of XAI performance metrics"""
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle('Explainable AI Performance Metrics', fontsize=16, fontweight='bold')
        
        # 1. Computation Time Distribution
        times = [m['time_seconds'] for m in metrics_list]
        axes[0, 0].hist(times, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
        axes[0, 0].set_title('Explanation Computation Time')
        axes[0, 0].set_xlabel('Time (seconds)')
        axes[0, 0].set_ylabel('Frequency')
        axes[0, 0].axvline(np.mean(times), color='red', linestyle='--', label=f'Mean: {np.mean(times):.3f}s')
        axes[0, 0].legend()
        
        # 2. Confidence Distribution
        confidences = [m['top_confidence'] for m in metrics_list]
        axes[0, 1].hist(confidences, bins=20, color='lightgreen', edgecolor='black', alpha=0.7)
        axes[0, 1].set_title('Top Prediction Confidence')
        axes[0, 1].set_xlabel('Confidence')
        axes[0, 1].set_ylabel('Frequency')
        axes[0, 1].axvline(np.mean(confidences), color='red', linestyle='--', label=f'Mean: {np.mean(confidences):.3f}')
        axes[0, 1].legend()
        
        # 3. Confidence Gap (Top 1 vs Top 2)
        gaps = [m['confidence_gap'] for m in metrics_list]
        axes[0, 2].hist(gaps, bins=20, color='lightsalmon', edgecolor='black', alpha=0.7)
        axes[0, 2].set_title('Confidence Gap (Top-1 vs Top-2)')
        axes[0, 2].set_xlabel('Confidence Gap')
        axes[0, 2].set_ylabel('Frequency')
        axes[0, 2].axvline(np.mean(gaps), color='red', linestyle='--', label=f'Mean: {np.mean(gaps):.3f}')
        axes[0, 2].legend()
        
        # 4. Attention Statistics
        att_means = [m['attention_mean'] for m in metrics_list]
        axes[1, 0].hist(att_means, bins=20, color='plum', edgecolor='black', alpha=0.7)
        axes[1, 0].set_title('Attention Weight Mean')
        axes[1, 0].set_xlabel('Mean Attention Weight')
        axes[1, 0].set_ylabel('Frequency')
        axes[1, 0].axvline(np.mean(att_means), color='red', linestyle='--', label=f'Mean: {np.mean(att_means):.4f}')
        axes[1, 0].legend()
        
        # 5. Top Token Importance
        top_weights = [m['top_token_weight'] for m in metrics_list]
        axes[1, 1].hist(top_weights, bins=20, color='khaki', edgecolor='black', alpha=0.7)
        axes[1, 1].set_title('Top Token Importance Weight')
        axes[1, 1].set_xlabel('Importance Weight')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].axvline(np.mean(top_weights), color='red', linestyle='--', label=f'Mean: {np.mean(top_weights):.4f}')
        axes[1, 1].legend()
        
        # 6. Time vs Confidence Scatter
        axes[1, 2].scatter(times, confidences, alpha=0.6, c=gaps, cmap='viridis')
        axes[1, 2].set_title('Computation Time vs Confidence')
        axes[1, 2].set_xlabel('Time (seconds)')
        axes[1, 2].set_ylabel('Confidence')
        axes[1, 2].grid(True, alpha=0.3)
        cbar = plt.colorbar(axes[1, 2].collections[0], ax=axes[1, 2])
        cbar.set_label('Confidence Gap')
        
        plt.tight_layout()
        return fig
    
    def compare_attention_patterns(self, text1, text2, label1="Sample 1", label2="Sample 2"):
        """Compare attention patterns between two samples"""
        
        tokens1, weights1 = self.explainer.get_attention_weights(text1)
        tokens2, weights2 = self.explainer.get_attention_weights(text2)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
        
        # Filter special tokens
        def filter_tokens(tokens, weights):
            valid = [(t, w) for t, w in zip(tokens, weights) if t not in ['[PAD]', '<pad>', '[CLS]', '[SEP]', '<s>', '</s>']]
            return [t for t, w in valid[:30]], [w for t, w in valid[:30]]
        
        t1, w1 = filter_tokens(tokens1, weights1)
        t2, w2 = filter_tokens(tokens2, weights2)
        
        # Plot 1
        ax1.bar(range(len(w1)), w1, color='cornflowerblue', alpha=0.7)
        ax1.set_title(f'Attention Pattern: {label1}', fontweight='bold')
        ax1.set_ylabel('Attention Weight')
        ax1.set_xticks(range(len(t1)))
        ax1.set_xticklabels([t.replace('▁', '').replace('Ġ', '') for t in t1], rotation=45, ha='right', fontsize=8)
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Plot 2
        ax2.bar(range(len(w2)), w2, color='coral', alpha=0.7)
        ax2.set_title(f'Attention Pattern: {label2}', fontweight='bold')
        ax2.set_ylabel('Attention Weight')
        ax2.set_xticks(range(len(t2)))
        ax2.set_xticklabels([t.replace('▁', '').replace('Ġ', '') for t in t2], rotation=45, ha='right', fontsize=8)
        ax2.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        return fig


def generate_xai_performance_report(analyzer, metrics_list, explanations_list, test_df):
    """Generate a comprehensive XAI performance report"""
    
    print("\n" + "=" * 70)
    print("COMPREHENSIVE XAI PERFORMANCE REPORT")
    print("=" * 70)
    
    # 1. Overall Statistics
    print("\n1. OVERALL STATISTICS")
    print("-" * 70)
    print(f"   Total samples analyzed: {len(metrics_list)}")
    print(f"   Total computation time: {sum(m['time_seconds'] for m in metrics_list):.2f}s")
    print(f"   Average time per explanation: {np.mean([m['time_seconds'] for m in metrics_list]):.3f}s")
    print(f"   Min time: {min(m['time_seconds'] for m in metrics_list):.3f}s")
    print(f"   Max time: {max(m['time_seconds'] for m in metrics_list):.3f}s")
    
    # 2. Prediction Quality
    print("\n2. PREDICTION QUALITY")
    print("-" * 70)
    confidences = [m['top_confidence'] for m in metrics_list]
    print(f"   Mean top confidence: {np.mean(confidences):.4f}")
    print(f"   Median top confidence: {np.median(confidences):.4f}")
    print(f"   Std dev: {np.std(confidences):.4f}")
    print(f"   Samples with >90% confidence: {sum(1 for c in confidences if c > 0.9)}/{len(confidences)}")
    print(f"   Samples with >80% confidence: {sum(1 for c in confidences if c > 0.8)}/{len(confidences)}")
    
    # 3. Attention Analysis
    print("\n3. ATTENTION ANALYSIS")
    print("-" * 70)
    att_means = [m['attention_mean'] for m in metrics_list]
    att_stds = [m['attention_std'] for m in metrics_list]
    print(f"   Average attention mean: {np.mean(att_means):.4f}")
    print(f"   Average attention std: {np.mean(att_stds):.4f}")
    print(f"   Attention concentration (mean std): {np.mean(att_stds):.4f}")
    
    # 4. Token Importance
    print("\n4. TOKEN IMPORTANCE")
    print("-" * 70)
    top_weights = [m['top_token_weight'] for m in metrics_list]
    print(f"   Average top token weight: {np.mean(top_weights):.4f}")
    print(f"   Max top token weight: {max(top_weights):.4f}")
    print(f"   Min top token weight: {min(top_weights):.4f}")
    
    # 5. Confidence Distribution
    print("\n5. CONFIDENCE DISTRIBUTION")
    print("-" * 70)
    high = sum(1 for c in confidences if c > 0.8)
    med = sum(1 for c in confidences if 0.5 < c <= 0.8)
    low = sum(1 for c in confidences if c <= 0.5)
    print(f"   High (>0.8):  {high:3d} samples ({high/len(confidences)*100:5.1f}%)")
    print(f"   Medium (0.5-0.8): {med:3d} samples ({med/len(confidences)*100:5.1f}%)")
    print(f"   Low (<=0.5):  {low:3d} samples ({low/len(confidences)*100:5.1f}%)")
    
    # 6. Most Common Predictions
    print("\n6. TOP PREDICTED MISCONCEPTIONS")
    print("-" * 70)
    all_predictions = [e['predictions'][0]['label'] for e in explanations_list]
    pred_counts = Counter(all_predictions)
    for pred, count in pred_counts.most_common(10):
        print(f"   {count:3d}x | {pred[:60]}...")
    
    print("\n" + "=" * 70)

## Section 5.2: Run XAI Performance Analysis

Test the performance analyzer on a batch of samples

In [ ]:
# Create performance analyzer
xai_analyzer = XAIPerformanceAnalyzer(explainer)

# Analyze a batch of test samples (adjust number as needed)
num_samples_to_analyze = min(50, len(test_df))  # Analyze up to 50 samples
test_texts = [create_input_text(row) for _, row in test_df.head(num_samples_to_analyze).iterrows()]

# Run batch analysis
xai_metrics, xai_explanations = xai_analyzer.analyze_batch(test_texts)

# Generate comprehensive report
generate_xai_performance_report(xai_analyzer, xai_metrics, xai_explanations, test_df)

## Section 5.3: Visualize XAI Performance

In [ ]:
# Plot performance metrics dashboard
perf_fig = xai_analyzer.plot_performance_metrics(xai_metrics)
plt.show()

print("\n✅ XAI Performance visualization complete!")

## Section 5.4: Compare Attention Patterns

In [ ]:
# Compare attention patterns between two different samples
if len(test_texts) >= 2:
    comparison_fig = xai_analyzer.compare_attention_patterns(
        test_texts[0], 
        test_texts[1],
        label1='Test Sample #1',
        label2='Test Sample #2'
    )
    plt.show()
    print("\n✅ Attention pattern comparison complete!")

## Section 6: Main Inference Pipeline

Detailed performance metrics and analysis for Explainable AI features.

In [ ]:
def main():
    """Main inference function"""
    
    print("\n" + "=" * 60)
    print("DeBERTa-v3-Large Inference with Explainable AI")
    print("=" * 60)
    
    # Initialize
    config = Config()
    seed_everything(config.SEED)
    print_device_info()
    
    # Load test data
    test_df = load_test_data(config)
    
    # Load model from checkpoint
    model, tokenizer, label_encoder = load_model_from_checkpoint(config)
    
    # Create Explainable AI module
    explainer = create_explainer(model, tokenizer, label_encoder, config)
    
    # Run XAI demonstration on test samples
    demo_explainer(explainer, test_df, num_samples=config.NUM_XAI_EXAMPLES)
    
    print("\n" + "=" * 60)
    print("✅ INFERENCE PIPELINE COMPLETE!")
    print("=" * 60)
    print("\nYou can now use the 'explainer' object to analyze any student response!")
    print("Example: explainer.explain_single_prediction(question, answer, explanation)")
    
    return model, tokenizer, label_encoder, explainer, test_df

## Run Inference Pipeline

In [ ]:
# Execute the main inference pipeline
model, tokenizer, label_encoder, explainer, test_df = main()

## Interactive Examples: Test Explainable AI

Use the explainer to analyze student misconceptions on test data!

In [ ]:
# Example 1: Explain a test sample from dataset
if len(test_df) > 0:
    sample_row = test_df.iloc[0]
    explanation = explainer.explain_single_prediction(
        question=sample_row['QuestionText'],
        student_answer=sample_row['MC_Answer'],
        student_explanation=sample_row['StudentExplanation']
    )
    
    print("\n✅ Explanation generated!")
    print("Top prediction:", explanation['predictions'][0]['label'])
    print("Confidence:", f"{explanation['predictions'][0]['probability']*100:.1f}%")

In [ ]:
# Example 2: Visualize attention for a test sample
if len(test_df) > 0:
    sample_text = create_input_text(test_df.iloc[0])
    fig = explainer.visualize_attention(sample_text)
    plt.show()

In [ ]:
# Example 3: Get token importance for any test input
if len(test_df) > 0:
    sample_text = create_input_text(test_df.iloc[0])
    important_tokens = explainer.highlight_important_tokens(sample_text, top_k=10)
    
    print("\nMost Important Tokens:")
    print("=" * 50)
    for i, token_info in enumerate(important_tokens, 1):
        print(f"{i}. '{token_info['token']}' - Importance: {token_info['weight']:.4f}")

## Custom Input Testing

Test the explainer with your own custom student responses:

In [ ]:
# Test with custom student response
custom_question = "What is 2 + 2 * 3?"
custom_answer = "12"
custom_explanation = "I added 2 + 2 to get 4, then multiplied by 3 to get 12"

explanation = explainer.explain_single_prediction(
    question=custom_question,
    student_answer=custom_answer,
    student_explanation=custom_explanation
)

print("\nCustom Input Analysis:")
print("=" * 70)
print(f"Question: {custom_question}")
print(f"Student Answer: {custom_answer}")
print(f"Explanation: {custom_explanation}")
print("\nTop 3 Predictions:")
for i, pred in enumerate(explanation['predictions'], 1):
    print(f"{i}. {pred['label'][:60]}... ({pred['probability']*100:.1f}%)")

## Summary

This inference notebook provides:
1. ✅ **DeBERTa-v3-Large inference** - optimized for Kaggle GPU  
2. ✅ **Explainable AI module** with attention visualization and human-readable explanations
3. ✅ **Detailed XAI Performance Analysis** - comprehensive metrics and benchmarks
4. ✅ **Interactive testing** - analyze any student response
5. ✅ **Performance Dashboards** - visual analytics for XAI quality

### Key Methods:
- `explainer.explain_single_prediction(question, answer, explanation)` - Explain any student response
- `explainer.visualize_attention(text)` - Show attention heatmap
- `explainer.highlight_important_tokens(text)` - Get key words that influenced prediction
- `explainer.generate_human_explanation(text)` - Comprehensive explanation with reasoning

### XAI Performance Analysis:
- `xai_analyzer.analyze_single_explanation(text)` - Analyze one explanation with metrics
- `xai_analyzer.analyze_batch(texts)` - Batch analysis with aggregated statistics
- `xai_analyzer.plot_performance_metrics(metrics)` - Visual performance dashboard
- `xai_analyzer.compare_attention_patterns(text1, text2)` - Compare attention between samples
- `generate_xai_performance_report(...)` - Comprehensive textual report

### Performance Metrics Tracked:
- ⏱️ Computation time and throughput
- 🎯 Prediction confidence distribution
- 🧠 Attention weight statistics
- 📊 Token importance analysis
- 📈 Confidence gaps and quality metrics

### Requirements for Kaggle:
- Upload trained model checkpoint as Kaggle dataset
- Update `MODEL_DIR` path in Config to point to your model dataset
- Test data should be in `/kaggle/input/dataset2/test.csv`
- GPU recommended for faster inference

Happy explaining! 🚀